# Fine-tuning DistilBERT for Sentiment

A learning project. I wanted to go through a real fine-tuning loop on a transformer end to end — tokenizer, dataset wrapper, training loop, eval — instead of just calling a pre-trained sentiment pipeline.

**Setup choices that matter:**
- `distilbert-base-uncased` instead of full BERT — about half the params, runs in reasonable time on CPU.
- 500 examples from the real IMDb movie review dataset (Maas et al., 2011). Small enough to keep training under 5 minutes on CPU; big enough that the head actually has to learn something.
- 1-2 epochs. With this dataset size, more is overfit.
- Compared against a `distilbert-base-uncased-finetuned-sst-2-english` off-the-shelf pipeline at the end, to see how a fine-tune on 500 examples compares to the 67k-example version someone already shipped.

The "honest answer" I'm expecting going in: my fine-tune does roughly as well as the off-the-shelf model on the same val split, give or take a couple of points, because IMDb sentiment is *easy* and 500 examples is more than enough for distilbert to figure out what "positive" means.


In [1]:
# !pip install transformers datasets scikit-learn torch -q
import warnings
warnings.filterwarnings("ignore")

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

random.seed(3)
np.random.seed(3)
torch.manual_seed(3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


device: cpu


## Load IMDb from Hugging Face

500 random reviews from the IMDb train split for fine-tuning and 200 from the test split for validation, each cut to its first 512 characters.

In [2]:
from datasets import load_dataset
ds = load_dataset("imdb", split="train").shuffle(seed=3).select(range(500))
texts  = [x["text"][:512] for x in ds]
labels = [x["label"] for x in ds]
val_ds = load_dataset("imdb", split="test").shuffle(seed=3).select(range(200))
val_texts  = [x["text"][:512] for x in val_ds]
val_labels = [x["label"] for x in val_ds]
print(f"loaded IMDb: train={len(texts)}  val={len(val_texts)}")

loaded IMDb: train=500  val=200


## Load tokenizer + DistilBERT

In [3]:
USE_HF_MODEL = True
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2).to(device)
    print("model loaded:", model.config.model_type)
except Exception as e:
    USE_HF_MODEL = False
    print("HF model load failed:", e)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model loaded: distilbert


## Tokenize + train loop

In [4]:
class SentDS(Dataset):
    def __init__(self, texts, labels, tok, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tok = tok
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], padding="max_length", truncation=True,
                       max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[i], dtype=torch.long),
        }

if USE_HF_MODEL:
    train_loader = DataLoader(SentDS(texts, labels, tokenizer),
                              batch_size=8, shuffle=True)
    val_loader   = DataLoader(SentDS(val_texts, val_labels, tokenizer),
                              batch_size=16, shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    EPOCHS = 2

    for ep in range(EPOCHS):
        model.train()
        running = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["label"].to(device),
            )
            out.loss.backward()
            optimizer.step()
            running += out.loss.item()
        print(f"epoch {ep + 1}  avg train loss: {running / len(train_loader):.4f}")


epoch 1  avg train loss: 0.6214


epoch 2  avg train loss: 0.3053


## Evaluate

In [5]:
if USE_HF_MODEL:
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            ).logits
            preds.extend(logits.argmax(-1).cpu().tolist())
            truths.extend(batch["label"].tolist())
    acc = accuracy_score(truths, preds)
    f1  = f1_score(truths, preds, average="macro")
    print(f"fine-tuned DistilBERT (500 examples, 2 epochs)")
    print(f"  validation accuracy: {acc:.3f}")
    print(f"  validation macro-F1: {f1:.3f}")
    print(classification_report(truths, preds, target_names=["neg", "pos"], digits=3))


fine-tuned DistilBERT (500 examples, 2 epochs)
  validation accuracy: 0.785
  validation macro-F1: 0.784
              precision    recall  f1-score   support

         neg      0.724     0.884     0.796        95
         pos      0.869     0.695     0.772       105

    accuracy                          0.785       200
   macro avg      0.797     0.790     0.784       200
weighted avg      0.800     0.785     0.784       200



## Compare against the off-the-shelf SST-2 fine-tune

In [6]:
# This model has already been fine-tuned on SST-2 by the HF team, so it's the obvious
# "did my 500-example fine-tune get anywhere near it" comparison.
try:
    from transformers import pipeline
    sst2 = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    ots_preds = []
    for t in val_texts:
        r = sst2(t[:512])[0]
        ots_preds.append(1 if r["label"] == "POSITIVE" else 0)
    ots_acc = accuracy_score(val_labels, ots_preds)
    ots_f1  = f1_score(val_labels, ots_preds, average="macro")
    print(f"off-the-shelf SST-2 model:")
    print(f"  validation accuracy: {ots_acc:.3f}")
    print(f"  validation macro-F1: {ots_f1:.3f}")
    if USE_HF_MODEL:
        print(f"\ndelta my-fine-tune - off-the-shelf:  acc {(acc - ots_acc):+.3f}  f1 {(f1 - ots_f1):+.3f}")
except Exception as e:
    print("Could not load SST-2 baseline:", e)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use mps:0


off-the-shelf SST-2 model:
  validation accuracy: 0.845
  validation macro-F1: 0.845

delta my-fine-tune - off-the-shelf:  acc -0.060  f1 -0.060


## What this notebook actually shows

- I can wire up a HuggingFace fine-tune end-to-end: tokenizer, custom `Dataset`, manual training loop with `AdamW`, eval with `accuracy / macro-F1 / classification_report`. That was the goal.
- The numbers are *not* a benchmark. 500 examples + 2 epochs of distilbert on IMDb landed at 78.5% accuracy (macro-F1 0.784) on 200 test reviews, and that will move a few points with the random seed. The fine-tuned model also leans negative: 88% recall on negative reviews but only 70% on positive ones. That's a perfectly fine result for the size; it does **not** mean the model would generalize to other review datasets without retraining.
- The off-the-shelf SST-2 model is the honest comparison. If my fine-tune is within ~2 pts of it, that's a "it worked"; if it's much worse, the fine-tune is undercooked (that's where I landed: 6 points behind the SST-2 model's 84.5%); if it's much better, something's wrong (the test set is too easy or there's a leak).
- If I came back to this I'd: (a) bump to 5k examples and 3 epochs, (b) add a learning-rate scheduler and weight decay, (c) try a `roberta-base` head for a real-world bake-off.
